# PPO Agent Testing Guide on MountainCar-v0

- Proximal Policy Optimization is also an actor-critic based architecture, it just solve the existing problem in actor-critic methods like A2C, A3C — Policy Collapse.
- So why can a policy collapse during training. Well, this happens when in a update step, we update the policy too much such that the new policy distribution is too far from the old policy distribution $\rightarrow$ you think it's normal? No, this means the agent forget all about the knowledge it's has learned and that's policy collapse.

This notebook provides a minimal workflow for testing a Proximal Policy Optimization (PPO) agent with a discrete action space. It configures the experiment, trains the agent, and prints the metrics returned by `learn()`. The cells are intended to be edited and rerun while experimenting with the agent and its hyperparameters.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import gymnasium as gym
import deep_rl

DEEP_RL_DIR = Path(deep_rl.__file__).resolve().parent
POLICY_GRADIENT_DIR = DEEP_RL_DIR / 'policy_gradient'  # runs/ output lives here

from deep_rl.policy_gradient.ppo import PPO

## 1. Configure the Experiment

Adjust these values before running the training cell. Start with a small `MAX_STEPS` while debugging, then increase it for longer experiments.

In [3]:
ENV_ID = 'MountainCar-v0'
SEED = 2026
MAX_STEPS = 20_000
HIDDEN_DIMS = (128, 128)
ACTOR_LR = 3e-4
CRITIC_LR = 1e-3
GAMMA = 0.99
GAE_LAMBDA = 0.95
CLIP_EPSILON = 0.2
ROLLOUT_STEPS = 128
UPDATE_EPOCHS = 10
BATCH_SIZE = 64
ENTROPY_COEF = 0.01
MAX_GRAD_NORM = 0.5

print(f'Environment: {ENV_ID}')
print(f'Seed: {SEED}')
print(f'Total steps: {MAX_STEPS}')
print(f'Rollout steps: {ROLLOUT_STEPS}')
print(f'Update epochs: {UPDATE_EPOCHS}')
print(f'Batch size: {BATCH_SIZE}')

Environment: MountainCar-v0
Seed: 2026
Total steps: 20000
Rollout steps: 128
Update epochs: 10
Batch size: 64


## 2. Create the Environment and Test the Agent

This cell creates a single environment, initializes PPO, runs training, and stores the returned metrics in `metrics`. The current PPO implementation collects a single-environment rollout, so the environment is not vectorized here. Rerun this cell after changing the configuration above.

In [4]:
env = gym.make(ENV_ID)
env.reset(seed=SEED)
env.action_space.seed(SEED)

state_dim = env.observation_space.shape[0]
action_dim = env.action_space.n

agent = PPO(
    state_dim=state_dim,
    action_dim=action_dim,
    hidden_dims=HIDDEN_DIMS,
    actor_lr=ACTOR_LR,
    critic_lr=CRITIC_LR,
    gamma=GAMMA,
    gae_lambda=GAE_LAMBDA,
    clip_epsilon=CLIP_EPSILON,
    rollout_steps=ROLLOUT_STEPS,
    update_epochs=UPDATE_EPOCHS,
    batch_size=BATCH_SIZE,
    entropy_coef=ENTROPY_COEF,
    max_grad_norm=MAX_GRAD_NORM,
    seed=SEED,
)

print(f'State dim: {state_dim}, Action dim: {action_dim}')
print(f'Starting training for {MAX_STEPS} steps...')
metrics = agent.learn(env, max_steps=MAX_STEPS)
env.close()
print('Training metrics:')
for name, value in metrics.items():
    print(f'  {name}: {value}')

State dim: 2, Action dim: 3
Starting training for 20000 steps...
Training metrics:
  policy_loss: -0.012848068773746491
  value_loss: 0.0671333321428392
  entropy: 0.005453448183834553


## 3. Inspect the Training Metrics

`PPO.learn()` returns aggregate information from the latest rollout update. The next cell prints a short explanation for each metric.

In [5]:
metric_descriptions = {
    'policy_loss': 'Clipped PPO policy objective.',
    'value_loss': 'Value-function prediction error.',
    'entropy': 'Policy uncertainty and exploration level.',
}

for name, description in metric_descriptions.items():
    print(f'{name}: {metrics[name]}')
    print(f'  {description}')

policy_loss: -0.012848068773746491
  Clipped PPO policy objective.
value_loss: 0.0671333321428392
  Value-function prediction error.
entropy: 0.005453448183834553
  Policy uncertainty and exploration level.


## 4. Hyperparameter Tuning

PPO is an improved version of the Actor-Critic method, which adds some techniques to keep the policy updates stable and make better use of the collected data. These techniques are:
- Using a clipped objective to limit how much the policy can change in one update
- Using GAE to compute the advantage, which balances bias and variance
- Reusing the same rollout for several epochs, split into mini-batches
- Often adding entropy regularization to encourage exploration

So the hyperparameters we need to choose carefully for this agent are similar to the classic Actor-Critic method, plus some PPO-specific ones:
- The learning rate for actor and critic. The actor learning rate is often smaller than the critic one (e.g. `3e-4` vs `1e-3`) -> because we want the critic to fit the returns quickly to give a stable signal for the policy updates.
- Actor and Critic network, we should start with a small network and only increase it if the task needs more capacity.
- The clip epsilon, which limits how far the policy can move in one update. A common starting point is `0.2`.
- The GAE lambda and discount factor, which control the bias-variance trade-off and how much the agent cares about future rewards. Common values are `0.95` and close to `1`.
- The rollout steps, update epochs and batch size. They affect each other, so we should choose them together. Note that the rollout can be shorter than `rollout_steps` if the episode ends early.
- The entropy coefficient, which controls exploration. Too large and the agent stays random, too small and the exploration stops too early.